<a href="https://colab.research.google.com/github/sowmya-prabha/Weight_compression_Algorithms/blob/main/CNN_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Opendatasets --quiet

In [2]:
import opendatasets as od

In [3]:
dataset_url = "https://www.kaggle.com/datasets/hojjatk/mnist-dataset/data"
od.download(dataset_url)

Skipping, found downloaded files in "./mnist-dataset" (use force=True to force download)


In [5]:
!pip install tensorflow --quiet

In [6]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [7]:
import tensorflow as tf

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()


In [8]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense
)

In [9]:
import pandas as pd

optimizers = [
    'adam',
    'sgd',
    'rmsprop',
    'adagrad',
    'adamax',
    'nadam'
]

results = []

for opt in optimizers:

    print("="*60)
    print("Optimizer :", opt)

    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=5,
        batch_size=128,
        validation_split=0.2,
        verbose=0
    )

    loss, acc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    print("Loss :", round(loss,4))
    print("Accuracy :", round(acc,4))

    results.append([
        opt,
        round(loss,4),
        round(acc,4)
    ])

results = pd.DataFrame(
    results,
    columns=[
        "Optimizer",
        "Loss",
        "Accuracy"
    ]
)

print("\nFinal Comparison")
print(results)

Optimizer : adam


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Loss : 0.0739
Accuracy : 0.9792
Optimizer : sgd
Loss : 2.301
Accuracy : 0.1135
Optimizer : rmsprop
Loss : 0.0733
Accuracy : 0.9853
Optimizer : adagrad
Loss : 0.2133
Accuracy : 0.9531
Optimizer : adamax
Loss : 0.0709
Accuracy : 0.9793
Optimizer : nadam
Loss : 0.0559
Accuracy : 0.9847

Final Comparison
  Optimizer    Loss  Accuracy
0      adam  0.0739    0.9792
1       sgd  2.3010    0.1135
2   rmsprop  0.0733    0.9853
3   adagrad  0.2133    0.9531
4    adamax  0.0709    0.9793
5     nadam  0.0559    0.9847


In [12]:
import numpy as np

for layer in model.layers:

    weights = layer.get_weights()

    if len(weights) > 0:

        W, b = weights

        threshold = np.percentile(np.abs(W), 50)

        W[np.abs(W) < threshold] = 0

        layer.set_weights([W, b])